In [ ]:
import tensorflow as tf
print(tf.__version__)

from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Create a master folder in Google Drive for your project
PROJECT_DIR = "/content/drive/MyDrive/ECommerce_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f"Project directory ready at: {PROJECT_DIR}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import requests
import shutil

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

2.20.0


In [ ]:
df = pd.read_csv('/content/electronics_product.csv')
# removing rupee symbols and the commas
df['price_no_symbols'] = df['actual_price'].astype(str).str.replace('₹', '', regex=False)
df['price_no_symbols'] = df['price_no_symbols'].str.replace(',', '', regex=False)

# changing it to the numbers
df['clean_price_inr'] = pd.to_numeric(df['price_no_symbols'], errors='coerce')
df = df.dropna(subset=['clean_price_inr', 'image'])

# changing the price to us dollars
df['price_usd'] = df['clean_price_inr'] / 96.08
df['price_usd'] = df['price_usd'].round(2)


low_cutoff_usd = df['price_usd'].quantile(0.20)
high_cutoff_usd = df['price_usd'].quantile(0.80)

# separating budget and premium
budget_df = df[df['price_usd'] <= low_cutoff_usd].copy()
budget_df['label'] = 0  # Standard / Budget

premium_df = df[df['price_usd'] >= high_cutoff_usd].copy()
premium_df['label'] = 1  # Premium


# filtering out the middle rows
filtered_df = df[(df['price_usd'] <= low_cutoff_usd) | (df['price_usd'] >= high_cutoff_usd)].reset_index(drop=True)

filtered_df = filtered_df.drop_duplicates(subset=['image']).reset_index(drop=True)

def assign_label(price):
    if price >= high_cutoff_usd:
        return 1
    else:
        return 0

filtered_df['label'] = filtered_df['price_usd'].apply(assign_label)

filtered_df.head()

filtered_df['label'].dtype


dtype('int64')

In [ ]:
# downloading images

if os.path.exists("Premium"):
    shutil.rmtree("Premium")
if os.path.exists("Standard"):
    shutil.rmtree("Standard")

os.makedirs("Premium", exist_ok=True)
os.makedirs("Standard", exist_ok=True)

# 3. Download loop
for index, row in filtered_df.iterrows():
    url = row["image"]
    label = row["label"]  # Integer 1 (Premium) or 0 (Standard)

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            # Map 1 to Premium folder, 0 to Standard folder
            folder = "Premium" if label == 1 else "Standard"
            filename = f"{folder}/{index}.jpg"

            with open(filename, "wb") as f:
                f.write(response.content)

    except Exception:
        # Skip broken image links or timeout errors automatically
        pass

print("Finished downloading images cleanly!")

Finished downloading images cleanly!


In [ ]:
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set destination path in Google Drive
drive_dir = '/content/drive/MyDrive/ECommerce_Project'
os.makedirs(f"{drive_dir}/Premium", exist_ok=True)
os.makedirs(f"{drive_dir}/Standard", exist_ok=True)

# 3. Copy temporary folders to Drive
print("Backing up downloaded images to Google Drive...")
shutil.copytree('/content/Premium', f"{drive_dir}/Premium", dirs_exist_ok=True)
shutil.copytree('/content/Standard', f"{drive_dir}/Standard", dirs_exist_ok=True)

print("Done! Your images are permanently saved in Google Drive.")

Mounted at /content/drive
Backing up downloaded images to Google Drive...
Done! Your images are permanently saved in Google Drive.
